# Gate analysis (v2) - saturation of the learned fusion gate

Reproducibly documents that the learned fusion gate of the gated variant
**degenerates**: instead of a soft per-token blend it saturates to ~100% surface
or ~0% surface (a quasi-hard switch), and which side is chosen depends on the
seed.

Runs **purely from stored artefacts** (`cv/dual_view_gated_v2/`), **no training,
no test access**. It is based on the out-of-fold gate weights per sentence from
the gated CV run.

**Output:** `analysis/gate_saturation.csv`, `analysis/gate_weight_histogram.png`


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os
# --- locate the project root -------------------------------------------
# No hardcoded Drive path: take KUSA_ROOT if it is set, otherwise the first
# candidate that actually contains config.py. Works in Colab and locally.
import os, sys
_CANDIDATES = [
    os.environ.get("KUSA_ROOT", ""),
    "/content/drive/MyDrive/kusa",
    "/content/drive/MyDrive/v2_heldout",
    "/content/drive/MyDrive/google_colab/kusa/v2_heldout",
    os.getcwd(),
    os.path.dirname(os.getcwd()),
]
V2_ROOT = next((p for p in _CANDIDATES
                if p and os.path.isfile(os.path.join(p, "config.py"))), None)
assert V2_ROOT, ("config.py not found - set KUSA_ROOT to the project "
                 "directory, e.g. os.environ['KUSA_ROOT'] = '/content/drive/MyDrive/kusa'")
sys.path.insert(0, V2_ROOT)
print("project root:", V2_ROOT)
from config import *

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

VARIANT = "dual_view_gated_v2"
print("Gate analysis for:", VARIANT, "- pure post-processing, no training.")

In [ ]:
# Load per-sentence gate weights. oof_predictions.csv has fold + surface_weight;
# falls back to oof_surface_weights.csv (without fold) if needed.
gdir = cv_dir(VARIANT)
p_pred = os.path.join(gdir, "oof_predictions.csv")
p_sw   = os.path.join(gdir, "oof_surface_weights.csv")

if os.path.exists(p_pred) and "surface_weight" in pd.read_csv(p_pred, nrows=1).columns:
    df = pd.read_csv(p_pred, encoding="utf-8")
    src = p_pred
elif os.path.exists(p_sw):
    df = pd.read_csv(p_sw, encoding="utf-8")
    src = p_sw
else:
    raise FileNotFoundError(
        f"Neither {p_pred} nor {p_sw} found - the gated CV must have run.")

assert "surface_weight" in df.columns, "column surface_weight is missing"
has_fold = "fold" in df.columns
sw = df["surface_weight"].values.astype(float)
print(f"Geladen: {src}")
print(f"{len(sw)} sentences | fold column present: {has_fold}")

In [ ]:
# --- Saturation metrics ------------------------------------------------
# A truly soft gate would lie mostly between 0.1 and 0.9. Saturation =
# fraction at the extremes.
def metrics(w):
    w = np.asarray(w, float)
    return {
        "n": len(w),
        "mean_surface": round(float(w.mean()), 4),
        "std_surface": round(float(w.std()), 4),
        "pct_surface_saturated_gt0.9": round(100 * float((w > 0.9).mean()), 2),
        "pct_lemma_saturated_lt0.1":   round(100 * float((w < 0.1).mean()), 2),
        "pct_blended_0.1_0.9":         round(100 * float(((w >= 0.1) & (w <= 0.9)).mean()), 2),
    }

overall = metrics(sw)
print("===== Overall (out-of-fold) =====")
for k, v in overall.items():
    print(f"  {k:28s}: {v}")

rows = [{"fold": "overall", **overall}]
if has_fold:
    print("\n===== Mean surface weight per fold =====")
    for f in sorted(df["fold"].unique()):
        m = metrics(df.loc[df["fold"] == f, "surface_weight"].values)
        rows.append({"fold": int(f), **m})
        print(f"  Fold {int(f)+1}: surface {m['mean_surface']*100:6.2f}%  |  "
              f"lemma {100-m['mean_surface']*100:6.2f}%  |  "
              f"saturated {m['pct_surface_saturated_gt0.9']+m['pct_lemma_saturated_lt0.1']:.1f}%")

os.makedirs(ANALYSIS, exist_ok=True)
sat_df = pd.DataFrame(rows)
sat_df.to_csv(os.path.join(ANALYSIS, "gate_saturation.csv"), index=False, encoding="utf-8")

print("\n----- Interpretation -----")
sat_total = overall["pct_surface_saturated_gt0.9"] + overall["pct_lemma_saturated_lt0.1"]
print(f"* {sat_total:.1f}% of all sentences are at an extreme (>0.9 or <0.1) -")
print(f"  the gate barely blends softly; it switches almost hard.")
print(f"* Only {overall['pct_blended_0.1_0.9']:.1f}% lie in the true blend range 0.1-0.9.")
print(f"* The overall mean ({overall['mean_surface']*100:.1f}% surface) is therefore an")
print(f"  averaging artefact of bimodal extremes, NOT a real 'x% surface' blend ratio")
print(f"  (std over sentences {overall['std_surface']*100:.1f} pp confirms the bimodality).")
if has_fold:
    print(f"* The fold means jump between ~0% and ~100% -> the view choice is")
    print(f"  seed-dependent, not stably learned.")
print(f"* Architecture note: the CNN branch ALWAYS uses the pure surface")
print(f"  (cnn_input = surface_out); the gate only blends the BiLSTM path. Hence")
print(f"  even a 'full-lemma' saturation costs no performance.")

In [ ]:
# --- Plot: histogram + fold means ---------------------------------------
if has_fold:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.2))
else:
    fig, ax1 = plt.subplots(1, 1, figsize=(6.5, 4.2)); ax2 = None

ax1.hist(sw, bins=50, range=(0, 1), color="#4C72B0", edgecolor="white")
ax1.axvline(0.1, color="grey", ls="--", lw=1)
ax1.axvline(0.9, color="grey", ls="--", lw=1)
ax1.set_xlabel("Gate surface weight (0 = all lemma, 1 = all surface)")
ax1.set_ylabel("Number of sentences")
ax1.set_title("Distribution of gate weights (out-of-fold)\ntwo peaks at the extremes = saturation")

if ax2 is not None:
    folds = sorted(df["fold"].unique())
    means = [df.loc[df["fold"] == f, "surface_weight"].mean() for f in folds]
    ax2.bar([f"Fold {int(f)+1}" for f in folds], means, color="#55A868", edgecolor="white")
    ax2.axhline(0.5, color="grey", ls="--", lw=1)
    ax2.set_ylim(0, 1)
    ax2.set_ylabel("mean surface weight")
    ax2.set_title("Fold means: seed-dependent view choice\n(one fold flips to lemma)")

plt.tight_layout()
out_png = os.path.join(ANALYSIS, "gate_weight_histogram.png")
plt.savefig(out_png, dpi=150, bbox_inches="tight")
plt.show()
print("saved:", out_png)
print("saved:", os.path.join(ANALYSIS, "gate_saturation.csv"))

## Reading for the paper

If the contribution were a *learned, adaptive per-token fusion gate*, this
analysis contradicts the mechanism: the gate saturates to a seed-dependent,
quasi-hard view selection. Together with the test result (gated is approximately
equal to average, not significant) this yields the honest, defensible story:

> The dual-view fusion improves the baseline robustly; a learned gate does not
> significantly improve fixed averaging and collapses to a saturated,
> seed-dependent view selection.

If the soft-blend story should nevertheless become central, that would be an
architecture change (entropy/temperature regularization, a separate smaller
gate learning rate, gate dropout) - not an evaluation detail.
